# M0 · GPU readiness and experimental discipline

**Learning objective:** before any optimization claim, prove that (1) the GPU you are measuring on is the GPU you think it is, (2) your timers respect GPU asynchrony, and (3) every number you report has a declared origin.

Reference lesson: [M00 lesson](../course/M00_setup_and_discipline/LESSON.md) · Switch language: [中文](./00_experimental_discipline.zh-CN.ipynb)

## The three course modes

| Mode | What runs | Where | Use it for |
|---|---|---|---|
| `REFERENCE` | frozen committed evidence only (CPU, offline) | `evidence/` | learning the method, CPU CI, debugging |
| `GPU_CLASSROOM` | **your own bounded W7900 measurements** | `runs/` | the default lab experience on Radeon Cloud |
| `GPU_RESEARCH` | full-protocol physical A/B campaigns | `runs/` | capstone / research-grade reproduction |

Set `COURSE_MODE = "GPU_CLASSROOM"` in the first code cell, then
**Kernel → Restart & Run All**. A classroom exercise is a *bounded physical
experiment*: it proves the pipeline works on your GPU and gives a first
comparison — it is **not** a formal performance claim (that needs the
research protocol: repeated prompts × seeds, matched populations).
Historical numbers in `evidence/` are references to investigate, never a
substitute for your own measurement.

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="00")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 1 · Hardware admission — is this really a W7900?
The admission banner above already answered in GPU modes. The cell below records the physical device facts and the exact model files this course pins (size + pinned sha256 digests — the trust anchor lives in `lab/weights.py`, never in a live endpoint).

In [ ]:
from lab import weights as weights_mod

if session.mode != "REFERENCE":
    d = session.admission.detail
    _stamp(MEASURED_TODAY,
           f"device={d['device_name']} gfx={d['gfx_arch']} "
           f"vram={d['vram_used_mib']}/{d['vram_total_mib']} MiB hip={d['hip']}")
else:
    print("REFERENCE mode: physical admission is skipped by design; "
          "the frozen campaign device facts are in evidence/results/env/")
for state in weights_mod.weights_state():
    r = weights_mod.verify_weight(state["rel"])
    print(f"  {state['state']:8s} {state['rel'].split('/')[-1]}  ({r['reason']})")

## 2 · GPU asynchrony — why unsynchronized timing is wrong
GPUs return control before the work finishes. Reading a timer immediately after launching a kernel measures *launch*, not *execution*. The cell below demonstrates it physically in GPU modes (a large bf16 matmul), and with a CPU thread analogy in REFERENCE mode so the lesson is never skipped.

In [ ]:
import time

def demo_async_gpu():
    import torch
    torch.manual_seed(0)
    a = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(20):
        b = a @ a
    t_no_sync = time.perf_counter() - t0      # WRONG: measures launch queue
    torch.cuda.synchronize()                  # correct: wait for completion
    t_synced = time.perf_counter() - t0       # RIGHT: includes execution
    print(f"WITHOUT sync: {t_no_sync:.4f}s   WITH sync: {t_synced:.4f}s")
    assert t_synced >= t_no_sync
    _stamp(MEASURED_TODAY, f"sync gap = {t_synced - t_no_sync:.3f}s on this GPU")

def demo_async_cpu_threads():
    import threading
    done = threading.Event()
    result = {}
    def work():
        x = sum(i * i for i in range(3_000_000))
        result["x"] = x
        done.set()
    t0 = time.perf_counter()
    threading.Thread(target=work).start()
    t_no_join = time.perf_counter() - t0      # WRONG: worker still running
    done.wait()                               # correct: join first
    t_joined = time.perf_counter() - t0
    print(f"WITHOUT join: {t_no_join:.5f}s   WITH join: {t_joined:.5f}s")
    _stamp(ESTIMATE, "CPU-thread analogy of GPU asynchrony (not a GPU measurement)")

gpu_cell(demo_async_gpu) or demo_async_cpu_threads()

## 3 · Smoke vs cold vs warm — three different populations
A **smoke** run proves the pipeline executes (one shot, no warmup, no statistics). A **cold** run includes first-load costs (weights from disk, allocator growth). **Warm** runs are the comparable population for latency claims. Pooling them is a classic measurement bug. Check the frozen campaign for real examples of each kind:

In [ ]:
from lab import notebook_support as ns
import statistics
import json

kinds = {}
for d in ns.read_json("lab/reference_runs.json")["reference_run_dirs"]["B0"]:
    for f in sorted((ROOT / "evidence/results" / d).glob("*.json")):
        rec = json.load(open(f))
        kinds.setdefault(rec["run_kind"], 0)
        kinds[rec["run_kind"]] += len(rec.get("runs", []))
print("B0 reference populations:", kinds)
_stamp(HISTORICAL, "population labels above come from the frozen campaign")

## 4 · Your turn
1. Predict the warm/cold gap for B0 on your instance, then (in GPU_CLASSROOM) compare with M1.
2. Find in the frozen records one warm sample slower than the cold sample of another session — why does 'cold is slowest' fail as an invariant?
3. Explain why a smoke PASS can never establish a speedup.

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.